<a href="https://colab.research.google.com/github/nihemelandu/causal-price-demand/blob/main/notebooks/01_data_understanding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Cell 1 — Environment Setup and Data Loading

In [ ]:
# Cell 1: Environment Setup and Data Loading
# Purpose: Load FreshRetailNet-50K and perform first contact with the data
# CRISP-DM Phase 2: Collect Initial Data

from datasets import load_dataset
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load dataset from HuggingFace
print("Loading FreshRetailNet-50K...")
dataset = load_dataset("Dingdong-Inc/FreshRetailNet-50K")
print((dataset)

# Convert train split to pandas DataFrame
df = dataset['train'].to_pandas()

print("\n--- Basic Shape ---")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

print("\n--- Column Names and Data Types ---")
print(df.dtypes)

print("\n--- First 5 Rows ---")
print(df.head())

print("\n--- Last 5 Rows ---")
print(df.tail())

Loading FreshRetailNet-50K...
<class 'datasets.dataset_dict.DatasetDict'>

--- Basic Shape ---
Rows: 4,500,000
Columns: 19

--- Column Names and Data Types ---
city_id                  int64
store_id                 int64
management_group_id      int64
first_category_id        int64
second_category_id       int64
third_category_id        int64
product_id               int64
dt                      object
sale_amount            float64
hours_sale              object
stock_hour6_22_cnt       int32
hours_stock_status      object
discount               float64
holiday_flag             int32
activity_flag            int32
precpt                 float64
avg_temperature        float64
avg_humidity           float64
avg_wind_level         float64
dtype: object

--- First 5 Rows ---
   city_id  store_id  management_group_id  first_category_id  \
0        0         0                    0                  5   
1        0         0                    0                  5   
2        0         0   

## Cell 2 — Data Structure Verification

In [ ]:
# Cell 2: Data Structure Verification
# Purpose: Verify the panel structure (store x product x time)
# CRISP-DM Phase 2: Describe Data

print("=== PANEL STRUCTURE VERIFICATION ===\n")

# Unique counts across key dimensions
print("--- Unique Counts ---")
print(f"Unique cities:    {df['city_id'].nunique():,}")
print(f"Unique stores:    {df['store_id'].nunique():,}")
print(f"Unique products:  {df['product_id'].nunique():,}")
print(f"Unique dates:     {df['dt'].nunique():,}")
print(f"Date range:       {df['dt'].min()} to {df['dt'].max()}")

print("\n--- Panel Balance Check ---")
# Count observations per store-product pair
obs_per_pair = df.groupby(['store_id', 'product_id'])['dt'].count()
print(f"Total store-product pairs: {obs_per_pair.shape[0]:,}")
print(f"Min observations per pair: {obs_per_pair.min()}")
print(f"Max observations per pair: {obs_per_pair.max()}")
print(f"Mean observations per pair: {obs_per_pair.mean():.1f}")
print(f"Median observations per pair: {obs_per_pair.median():.1f}")
print(f"\nIs panel balanced? {obs_per_pair.nunique() == 1}")

print("\n--- Observations per Product (across all stores) ---")
obs_per_product = df.groupby('product_id')['dt'].count()
print(obs_per_product.describe())

print("\n--- Observations per Store (across all products) ---")
obs_per_store = df.groupby('store_id')['dt'].count()
print(obs_per_store.describe())

print("\n--- Product Coverage per Store ---")
# How many products does each store carry?
products_per_store = df.groupby('store_id')['product_id'].nunique()
print(products_per_store.describe())

print("\n--- Store Coverage per Product ---")
# How many stores carry each product?
stores_per_product = df.groupby('product_id')['store_id'].nunique()
print(stores_per_product.describe())

=== PANEL STRUCTURE VERIFICATION ===

--- Unique Counts ---
Unique cities:    18
Unique stores:    898
Unique products:  865
Unique dates:     90
Date range:       2024-03-28 to 2024-06-25

--- Panel Balance Check ---
Total store-product pairs: 50,000
Min observations per pair: 90
Max observations per pair: 90
Mean observations per pair: 90.0
Median observations per pair: 90.0

Is panel balanced? True

--- Observations per Product (across all stores) ---
count      865.000000
mean      5202.312139
std       9721.666808
min         90.000000
25%        180.000000
50%        900.000000
75%       5670.000000
max      70470.000000
Name: dt, dtype: float64

--- Observations per Store (across all products) ---
count      898.000000
mean      5011.135857
std       2813.220826
min         90.000000
25%       2700.000000
50%       4500.000000
75%       7020.000000
max      14760.000000
Name: dt, dtype: float64

--- Product Coverage per Store ---
count    898.000000
mean      55.679287
std      

## Cell 3 — Outcome Variable Analysis (sale_amount)

In [ ]:
# Cell 3: Outcome Variable Analysis
# Purpose: Understand the distribution and quality of the demand signal
# CRISP-DM Phase 2: Explore Data

print("=== OUTCOME VARIABLE: sale_amount ===\n")

print("--- Summary Statistics ---")
print(df['sale_amount'].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]))

print("\n--- Zero Sales Analysis ---")
zero_sales = df['sale_amount'] == 0
print(f"Total zero-sale records: {zero_sales.sum():,} ({zero_sales.mean()*100:.1f}%)")

print("\n--- Zero Sales vs Stockout Status ---")
# Are zeros due to stockouts or true zero demand?
# stock_hour6_22_cnt = 0 means out of stock all day
stockout_day = df['stock_hour6_22_cnt'] == 0
zero_with_stockout = (zero_sales & stockout_day).sum()
zero_without_stockout = (zero_sales & ~stockout_day).sum()
print(f"Zero sales WITH full stockout:    {zero_with_stockout:,} ({zero_with_stockout/zero_sales.sum()*100:.1f}% of zeros)")
print(f"Zero sales WITHOUT full stockout: {zero_without_stockout:,} ({zero_without_stockout/zero_sales.sum()*100:.1f}% of zeros)")

print("\n--- Stockout Rate Overall ---")
print(f"Days with full stockout (stock_hour6_22_cnt = 0): {stockout_day.sum():,} ({stockout_day.mean()*100:.1f}%)")

print("\n--- Demand Distribution by Product Category ---")
demand_by_cat = df.groupby('first_category_id')['sale_amount'].agg(['mean', 'median', 'std', 'count'])
print(demand_by_cat.sort_values('mean', ascending=False))

print("\n--- Temporal Demand Pattern ---")
df['dt'] = pd.to_datetime(df['dt'])
df['day_of_week'] = df['dt'].dt.dayofweek
demand_by_dow = df.groupby('day_of_week')['sale_amount'].mean()
days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
demand_by_dow.index = days
print("Average demand by day of week:")
print(demand_by_dow)

print("\n--- Monthly Demand Pattern ---")
df['month'] = df['dt'].dt.month
demand_by_month = df.groupby('month')['sale_amount'].mean()
print("Average demand by month:")
print(demand_by_month)

=== OUTCOME VARIABLE: sale_amount ===

--- Summary Statistics ---
count    4.500000e+06
mean     9.985913e-01
std      1.406738e+00
min      0.000000e+00
1%       0.000000e+00
5%       1.000000e-01
25%      4.000000e-01
50%      7.000000e-01
75%      1.100000e+00
95%      2.900000e+00
99%      5.800000e+00
max      4.490000e+01
Name: sale_amount, dtype: float64

--- Zero Sales Analysis ---
Total zero-sale records: 200,760 (4.5%)

--- Zero Sales vs Stockout Status ---
Zero sales WITH full stockout:    46,944 (23.4% of zeros)
Zero sales WITHOUT full stockout: 153,816 (76.6% of zeros)

--- Stockout Rate Overall ---
Days with full stockout (stock_hour6_22_cnt = 0): 2,507,994 (55.7%)

--- Demand Distribution by Product Category ---
                       mean  median       std   count
first_category_id                                    
21                 1.795060     0.6  4.051033  255870
29                 1.467059     1.0  1.644060  314640
20                 1.284182     0.9  1.373642  

## Cell 4 — Treatment Variable Analysis (discount)

In [ ]:
# Cell 4: Treatment Variable Analysis
# Purpose: Assess whether discount has sufficient variation for causal identification
# CRISP-DM Phase 2: Explore Data — this is the most critical cell for DML feasibility

print("=== TREATMENT VARIABLE: discount ===\n")

print("--- Summary Statistics ---")
print(df['discount'].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]))

print("\n--- Discount Value Distribution ---")
print(f"Unique discount values: {df['discount'].nunique()}")
print("\nTop 20 most frequent discount values:")
print(df['discount'].value_counts().head(20))

print("\n--- Discount = 1 (no discount) vs discounted ---")
no_discount = df['discount'] == 1
discounted = df['discount'] < 1
print(f"Records with no discount (discount=1): {no_discount.sum():,} ({no_discount.mean()*100:.1f}%)")
print(f"Records with discount (discount<1):    {discounted.sum():,} ({discounted.mean()*100:.1f}%)")

print("\n--- Discount Variation Across Products ---")
discount_by_product = df.groupby('product_id')['discount'].agg(['mean', 'std', 'min', 'max', 'nunique'])
print("Summary of per-product discount variation:")
print(discount_by_product.describe())

print("\n--- Products with NO price variation (discount never changes) ---")
no_variation = discount_by_product[discount_by_product['nunique'] == 1]
print(f"Products with constant discount: {len(no_variation):,} ({len(no_variation)/discount_by_product.shape[0]*100:.1f}%)")

print("\n--- Discount Variation Across Stores for Same Product ---")
# For DML we want within-product, within-store variation over time
within_variation = df.groupby(['store_id', 'product_id'])['discount'].std()
print("Within store-product discount std dev:")
print(within_variation.describe())
print(f"\nStore-product pairs with ZERO within variation: {(within_variation == 0).sum():,} ({(within_variation == 0).mean()*100:.1f}%)")

print("\n--- Discount and activity_flag alignment ---")
print("activity_flag distribution:")
print(df['activity_flag'].value_counts())
print("\nMean discount when activity_flag=1 vs 0:")
print(df.groupby('activity_flag')['discount'].mean())

print("\n--- Discount Variation Over Time ---")
discount_by_date = df.groupby('dt')['discount'].mean()
print("Daily mean discount (first 10 and last 10 days):")
print(discount_by_date.head(10))
print(discount_by_date.tail(10))

=== TREATMENT VARIABLE: discount ===

--- Summary Statistics ---
count    4.500000e+06
mean     9.111412e-01
std      1.281734e-01
min      0.000000e+00
1%       5.040000e-01
5%       6.660000e-01
25%      8.510000e-01
50%      9.890000e-01
75%      1.000000e+00
95%      1.000000e+00
99%      1.000000e+00
max      1.088000e+00
Name: discount, dtype: float64

--- Discount Value Distribution ---
Unique discount values: 2600

Top 20 most frequent discount values:
discount
1.000    2181121
0.909      28919
0.882      28327
0.833      28099
0.869      27957
0.750      25832
0.799      23089
0.908      22395
0.836      20806
0.850      20275
0.625      19841
0.881      19702
0.940      19014
0.666      18820
0.712      18014
0.950      17788
0.883      17652
0.913      17366
0.887      16061
0.000      16039
Name: count, dtype: int64

--- Discount = 1 (no discount) vs discounted ---
Records with no discount (discount=1): 2,181,121 (48.5%)
Records with discount (discount<1):    2,318,845 (51.

## Cell 5 — Confounder Coverage Analysis

In [ ]:
# Cell 5: Confounder Coverage Analysis
# Purpose: Assess whether available covariates provide adequate confounding control for DML
# CRISP-DM Phase 2: Verify Data Quality + Explore Data

print("=== CONFOUNDER COVERAGE ANALYSIS ===\n")

confounders = [
    'holiday_flag',
    'activity_flag',
    'precpt',
    'avg_temperature',
    'avg_humidity',
    'avg_wind_level',
    'stock_hour6_22_cnt'
]

print("--- Missing Values in Confounders ---")
for col in confounders:
    missing = df[col].isna().sum()
    print(f"{col}: {missing:,} missing ({missing/len(df)*100:.2f}%)")

print("\n--- Summary Statistics for Continuous Confounders ---")
continuous_confounders = ['precpt', 'avg_temperature', 'avg_humidity', 'avg_wind_level']
print(df[continuous_confounders].describe())

print("\n--- Binary Confounder Distributions ---")
binary_confounders = ['holiday_flag', 'activity_flag']
for col in binary_confounders:
    print(f"\n{col}:")
    print(df[col].value_counts())
    print(f"  Proportion = 1: {df[col].mean()*100:.1f}%")

print("\n--- Weather Variation Across Cities ---")
weather_by_city = df.groupby('city_id')[continuous_confounders].mean()
print("Mean weather values by city (first 10 cities):")
print(weather_by_city.head(10))

print("\n--- Do Confounders Vary Over Time? ---")
# Time variation is important for DML — confounders must vary
for col in continuous_confounders:
    time_std = df.groupby('dt')[col].mean().std()
    print(f"{col} — std of daily mean over time: {time_std:.4f}")

print("\n--- Product Hierarchy Coverage ---")
print(f"Unique management groups:   {df['management_group_id'].nunique()}")
print(f"Unique first categories:    {df['first_category_id'].nunique()}")
print(f"Unique second categories:   {df['second_category_id'].nunique()}")
print(f"Unique third categories:    {df['third_category_id'].nunique()}")
print(f"Unique products:            {df['product_id'].nunique()}")

print("\n--- Demand vs Key Confounders (correlation check) ---")
# Preliminary signal: do confounders correlate with demand?
for col in confounders:
    corr = df['sale_amount'].corr(df[col])
    print(f"Correlation of sale_amount with {col}: {corr:.4f}")

=== CONFOUNDER COVERAGE ANALYSIS ===

--- Missing Values in Confounders ---
holiday_flag: 0 missing (0.00%)
activity_flag: 0 missing (0.00%)
precpt: 0 missing (0.00%)
avg_temperature: 0 missing (0.00%)
avg_humidity: 0 missing (0.00%)
avg_wind_level: 0 missing (0.00%)
stock_hour6_22_cnt: 0 missing (0.00%)

--- Summary Statistics for Continuous Confounders ---
             precpt  avg_temperature  avg_humidity  avg_wind_level
count  4.500000e+06     4.500000e+06  4.500000e+06    4.500000e+06
mean   3.698385e+00     2.227911e+01  7.445046e+01    1.724542e+00
std    3.683960e+00     3.593264e+00  1.009831e+01    3.845731e-01
min    0.000000e+00     1.206000e+01  2.735000e+01    9.700000e-01
25%    1.428800e+00     1.939000e+01  7.162000e+01    1.450000e+00
50%    2.297700e+00     2.250000e+01  7.636000e+01    1.640000e+00
75%    4.481600e+00     2.540000e+01  8.071000e+01    1.950000e+00
max    4.250000e+01     3.088000e+01  1.000000e+02    3.850000e+00

--- Binary Confounder Distributions

## Cell 6 — Data Quality Verification

In [ ]:
# Cell 6: Data Quality Verification
# Purpose: Catalogue all data quality issues that must be resolved before modeling
# CRISP-DM Phase 2: Verify Data Quality

print("=== DATA QUALITY VERIFICATION ===\n")

print("--- Missing Values Across All Columns ---")
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({
    'missing_count': missing,
    'missing_pct': missing_pct
}).sort_values('missing_pct', ascending=False)
print(missing_report[missing_report['missing_count'] > 0])

print("\n--- Negative Values Check ---")
numeric_cols = df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    neg_count = (df[col] < 0).sum()
    if neg_count > 0:
        print(f"{col}: {neg_count:,} negative values")
print("(no output above means no negative values found)")

print("\n--- Duplicate Records Check ---")
dupes = df.duplicated(subset=['store_id', 'product_id', 'dt']).sum()
print(f"Duplicate store-product-date records: {dupes:,}")

print("\n--- Sparse SKUs (low observation count) ---")
obs_per_sku = df.groupby('product_id')['dt'].count()
sparse_threshold = 30  # fewer than 30 days of data
sparse_skus = obs_per_sku[obs_per_sku < sparse_threshold]
print(f"SKUs with fewer than {sparse_threshold} observations: {len(sparse_skus):,} ({len(sparse_skus)/obs_per_sku.shape[0]*100:.1f}%)")

print("\n--- Long-tail Demand Distribution ---")
# 20% of SKUs account for 51.8% of transactions per paper
demand_by_sku = df.groupby('product_id')['sale_amount'].sum().sort_values(ascending=False)
total_demand = demand_by_sku.sum()
top_20pct_skus = int(len(demand_by_sku) * 0.2)
top_20pct_demand = demand_by_sku.iloc[:top_20pct_skus].sum()
print(f"Top 20% of SKUs account for: {top_20pct_demand/total_demand*100:.1f}% of total demand")

print("\n--- Discount Variable Sanity Check ---")
print(f"Discount values > 1 (invalid): {(df['discount'] > 1).sum():,}")
print(f"Discount values < 0 (invalid): {(df['discount'] < 0).sum():,}")
print(f"Discount values = 0 (100% off — suspicious): {(df['discount'] == 0).sum():,}")

print("\n--- Date Continuity Check ---")
df['dt'] = pd.to_datetime(df['dt'])
date_range = pd.date_range(start=df['dt'].min(), end=df['dt'].max(), freq='D')
actual_dates = df['dt'].unique()
missing_dates = set(date_range) - set(actual_dates)
print(f"Expected date range: {df['dt'].min().date()} to {df['dt'].max().date()}")
print(f"Total expected dates: {len(date_range)}")
print(f"Actual unique dates in data: {len(actual_dates)}")
print(f"Missing dates: {len(missing_dates)}")
if missing_dates:
    print(sorted(missing_dates))

print("\n--- Hours Sale Sequence Sanity Check ---")
# hours_sale is a list of 24 hourly values — verify length
hour_lengths = df['hours_sale'].apply(len)
print(f"hours_sale sequence length — min: {hour_lengths.min()}, max: {hour_lengths.max()}, all 24: {(hour_lengths == 24).all()}")

# Verify hours_sale sums to sale_amount
df['hours_sale_sum'] = df['hours_sale'].apply(sum)
discrepancy = (df['hours_sale_sum'] - df['sale_amount']).abs()
print(f"\nhours_sale sum vs sale_amount — max discrepancy: {discrepancy.max():.6f}")
print(f"Records with discrepancy > 0.01: {(discrepancy > 0.01).sum():,}")

=== DATA QUALITY VERIFICATION ===

--- Missing Values Across All Columns ---
Empty DataFrame
Columns: [missing_count, missing_pct]
Index: []

--- Negative Values Check ---
(no output above means no negative values found)

--- Duplicate Records Check ---
Duplicate store-product-date records: 0

--- Sparse SKUs (low observation count) ---
SKUs with fewer than 30 observations: 0 (0.0%)

--- Long-tail Demand Distribution ---
Top 20% of SKUs account for: 87.0% of total demand

--- Discount Variable Sanity Check ---
Discount values > 1 (invalid): 34
Discount values < 0 (invalid): 0
Discount values = 0 (100% off — suspicious): 16,039

--- Date Continuity Check ---
Expected date range: 2024-03-28 to 2024-06-25
Total expected dates: 90
Actual unique dates in data: 90
Missing dates: 0

--- Hours Sale Sequence Sanity Check ---
hours_sale sequence length — min: 24, max: 24, all 24: True

hours_sale sum vs sale_amount — max discrepancy: 0.000000
Records with discrepancy > 0.01: 0


## Cell 7 — DML Feasibility Summary

In [ ]:
# Cell 7: DML Feasibility Summary
# Purpose: Synthesise findings into a structured fit-for-purpose assessment
# Bridges CRISP-DM Phase 2 findings toward Phase 3 planning

print("=== DML FEASIBILITY SUMMARY ===\n")

print("--- 1. Panel Structure ---")
n_products = df['product_id'].nunique()
n_stores = df['store_id'].nunique()
n_dates = df['dt'].nunique()
print(f"Panel dimensions: {n_stores} stores × {n_products} products × {n_dates} time periods")
print(f"Total store-product pairs: {df.groupby(['store_id','product_id']).ngroups:,}")

print("\n--- 2. Treatment Variable (discount) ---")
discount_variation = df.groupby(['store_id','product_id'])['discount'].std()
pct_with_variation = (discount_variation > 0).mean() * 100
print(f"Store-product pairs with price variation: {pct_with_variation:.1f}%")
print(f"Overall discount std dev: {df['discount'].std():.4f}")

print("\n--- 3. Outcome Variable (sale_amount) ---")
stockout_rate = (df['stock_hour6_22_cnt'] == 0).mean() * 100
print(f"Direct observation: YES")
print(f"Stockout-induced censoring rate: {stockout_rate:.1f}%")
print(f"Censoring correction required: {'YES' if stockout_rate > 5 else 'NO'}")

print("\n--- 4. Confounder Availability ---")
available_confounders = [
    'holiday_flag (temporal)',
    'activity_flag (promotional)',
    'discount (price variation signal)',
    'precpt, avg_temperature, avg_humidity, avg_wind_level (weather)',
    'stock_hour6_22_cnt / hours_stock_status (availability)',
    'first/second/third_category_id (product hierarchy)',
    'city_id, store_id (geographic fixed effects)'
]
print("Available confounders:")
for c in available_confounders:
    print(f"  ✓ {c}")

missing_confounders = [
    'Unit price (absolute) — only discount rate available',
    'Product quality signals (ratings, reviews) — not present',
    'Competitor pricing — not present',
    'Text/image product descriptions — not present'
]
print("\nMissing confounders (vs paper dataset):")
for c in missing_confounders:
    print(f"  ✗ {c}")

print("\n--- 5. Key Data Quality Issues Requiring Resolution ---")
issues = [
    "Demand censoring from stockouts — requires imputation or exclusion",
    "Long-tail SKU sparsity — requires minimum observation threshold",
    "Discount = 1 dominance — requires assessment of identification power",
    "No absolute price — treatment variable must be constructed from discount",
    "Three-month temporal scope — limits seasonal generalisation"
]
for i, issue in enumerate(issues, 1):
    print(f"  {i}. {issue}")

print("\n--- OVERALL VERDICT ---")
print("""
FreshRetailNet-50K IS fit for purpose for causal price elasticity
estimation using DML, subject to the following adaptations:

  1. Use discount as the treatment variable (promotional price variation)
  2. Correct for demand censoring using stockout annotations before modeling
  3. Apply a minimum observation threshold to exclude sparse SKUs
  4. Aggregate to daily or weekly level for DML panel estimation
  5. Use product hierarchy + weather + holiday flags as confounders
  6. Apply store and product fixed effects to absorb time-invariant confounding

These are engineering and analytical choices, not fundamental obstacles.
The dataset's panel richness, directly observed demand, and rich confounder
set make it well-suited to the DML methodology in the anchor paper.
""")

=== DML FEASIBILITY SUMMARY ===

--- 1. Panel Structure ---
Panel dimensions: 898 stores × 865 products × 90 time periods
Total store-product pairs: 50,000

--- 2. Treatment Variable (discount) ---
Store-product pairs with price variation: 96.8%
Overall discount std dev: 0.1282

--- 3. Outcome Variable (sale_amount) ---
Direct observation: YES
Stockout-induced censoring rate: 55.7%
Censoring correction required: YES

--- 4. Confounder Availability ---
Available confounders:
  ✓ holiday_flag (temporal)
  ✓ activity_flag (promotional)
  ✓ discount (price variation signal)
  ✓ precpt, avg_temperature, avg_humidity, avg_wind_level (weather)
  ✓ stock_hour6_22_cnt / hours_stock_status (availability)
  ✓ first/second/third_category_id (product hierarchy)
  ✓ city_id, store_id (geographic fixed effects)

Missing confounders (vs paper dataset):
  ✗ Unit price (absolute) — only discount rate available
  ✗ Product quality signals (ratings, reviews) — not present
  ✗ Competitor pricing — not pres